In [ ]:
import os
import ast
import json
import pickle
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
import cartopy.crs as ccrs
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
RAWDIR     = CONFIGS['filepaths']['raw']
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELS     = CONFIGS['experiments']
SPLIT      = 'test'

FIELDVARS    = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS      = MODELS['nn']['seeds']
OPTIMIZEDEQS = MODELS['sr']['optimizedeqs']
PODRUNS      = MODELS['pod']['runs']
NNRUNS       = MODELS['nn']['runs']

SRFUNCTIONS = {
    'cube':lambda x:x**3, 'square':lambda x:x**2, 'neg':lambda x:-x,
    'sqrt':np.sqrt, 'exp':np.exp, 'log':np.log, 'abs':np.abs,
    'max':np.maximum, 'min':np.minimum}

COLORS = {}
LABELS = {}
for name,config in {**PODRUNS,**NNRUNS,**OPTIMIZEDEQS}.items():
    COLORS[name] = config['color']
    LABELS[name] = config['description']

In [ ]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None: w = w*mask[:,None,:]
    return w.sum(axis=2)

def physical_prediction(output):
    return np.expm1(tpstd*np.maximum(0.0,np.asarray(output,dtype=float)))

def eval_form(form,variables,constants):
    ns = dict(SRFUNCTIONS)
    ns.update(variables)
    ns.update(constants)
    return np.asarray(eval(form,{'__builtins__':{}},ns),dtype=float)

def used_predictors(form,candidates):
    names = {n.id for n in ast.walk(ast.parse(form,mode='eval')) if isinstance(n,ast.Name)}
    return [c for c in candidates if c in names]

def r2(obs,pred):
    mask = np.isfinite(obs)&np.isfinite(pred)
    o,p  = obs[mask],pred[mask]
    return 1-np.sum((o-p)**2)/np.sum((o-o.mean())**2)

def bin_1d(x,z,nbins=20,minsamples=50,plo=1,phi=99):
    finite = np.isfinite(x)&np.isfinite(z)
    x,z    = x[finite],z[finite]
    lo,hi  = np.percentile(x,[plo,phi])
    edges  = np.linspace(lo,hi,nbins+1)
    xidxs  = np.clip(np.digitize(x,edges)-1,0,nbins-1)
    means  = np.full(nbins,np.nan)
    counts = np.zeros(nbins,dtype=int)
    for i in range(nbins):
        idx = xidxs==i
        counts[i] = idx.sum()
        if counts[i]>=minsamples: means[i] = z[idx].mean()
    return 0.5*(edges[:-1]+edges[1:]),means,counts

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
tpmean = float(STATS['tp_mean'])
tpstd  = float(STATS['tp_std'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nlat  = ds.sizes['lat']
    nlon  = ds.sizes['lon']
    nsig  = ds.sizes.get('sig',1)
    lat   = ds['lat'].values
    lon   = ds['lon'].values
    dsig  = ds['dsig'].values
    farrs      = [ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS]
    fieldstack = np.stack(farrs,axis=1)
    surfmask   = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                  if 'surfmask' in ds else None)
    def getflat(da):
        if 'time' in da.dims:
            return da.transpose('time','lat','lon').values.ravel()
        return np.tile(da.values,(ntime,1,1)).ravel()
    blnorm  = getflat(ds['bl'])
    lfnorm  = getflat(ds['lf'])
    shfnorm = getflat(ds['shf'])
    lhfnorm = getflat(ds['lhf'])
    sdonorm = getflat(ds['sdo'])   # std dev of orography (normalized)

kwlist = []
for seed in NNSEEDS:
    wpath = os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath,engine='h5netcdf') as wds:
            kwlist.append(wds['k'].values)
ki = np.mean([kernel_integrate(fieldstack,kw,dsig,surfmask) for kw in kwlist],axis=0) if kwlist else fieldstack.mean(axis=2)
rhk,thetaek,thetaestark = ki[:,0],ki[:,1],ki[:,2]

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsflat = ds.tp.transpose('time','lat','lon').values.ravel()
    sdoraw  = getflat(ds['sdo'])   # physical units (m)

# lat/lon for every sample
latidx = np.tile(np.arange(nlat)[None,:,None],(ntime,1,nlon)).ravel()
lonidx = np.tile(np.arange(nlon)[None,None,:],(ntime,nlat,1)).ravel()
lats   = lat[latidx]
lons   = lon[lonidx]

with open(os.path.join(MODELSDIR,'sr','optimized_equations.pkl'),'rb') as f:
    SR_REGISTRY = pickle.load(f)

## Average Surface Height (ASH)

SDO captures terrain **roughness** (sub-grid variability), but not mean elevation. ERA5 provides a static surface geopotential `z` (J/kg); dividing by g=9.80665 m/s² gives mean surface height in metres. This is distinct from SDO:

- **ASH**: how high the terrain is → controls large-scale forced ascent
- **SDO**: how rugged the terrain is → controls sub-grid channelling and wake effects

Both are standardised before entering models, so they appear as dimensionless z-scores in equations.

If `ERA5_orography` is not present in RAWDIR, ASH can be downloaded from the [ERA5 geopotential field](https://cds.climate.copernicus.eu/) (single level, invariant) or from ETOPO/GMTED via `pooch` / `rioxarray`. The cell below tries ERA5 first.

In [ ]:
G = 9.80665  # m/s²
ASH_MAP = None  # (nlat, nlon) mean surface height in metres

# Try ERA5 static geopotential (invariant field)
for candidate in ['ERA5_orography.nc','ERA5_geopotential.nc',
                   'ERA5_surface_geopotential.nc','ERA5_z.nc']:
    fpath = os.path.join(RAWDIR,candidate)
    if os.path.exists(fpath):
        with xr.open_dataset(fpath) as ds:
            zvar = [v for v in ds.data_vars if v in ('z','oro','orog','geopotential')][0]
            z2d  = ds[zvar].squeeze()
            # Subset and regrid to our lat/lon grid
            z2d  = z2d.sel(latitude=slice(lat.max()+1,lat.min()-1),
                           longitude=slice(lon.min()-1,lon.max()+1))
            z2d  = z2d.interp(latitude=lat[::-1],longitude=lon).values[::-1] / G
            ASH_MAP = z2d.astype(np.float32)
        print(f'Loaded ASH from {candidate}: range {ASH_MAP.min():.0f}–{ASH_MAP.max():.0f} m')
        break

if ASH_MAP is None:
    print('ERA5 orography not found. Using SDO as the sole orographic metric.')
    print('To add ASH: download ERA5 single-level "Geopotential" (invariant) from CDS')
    print('  and save as ERA5_orography.nc in RAWDIR.')

# SDO static map (time-invariant — take first time step)
SDO_MAP = sdoraw.reshape(ntime,nlat,nlon)[0]  # (nlat,nlon) in physical units
ASH_FLAT = (np.tile(ASH_MAP,(ntime,1,1)).ravel() if ASH_MAP is not None else None)

In [ ]:
VARS = {'bl':blnorm,'rh':rhk,'thetae':thetaek,'thetaestar':thetaestark,
        'lf':lfnorm,'shf':shfnorm,'lhf':lhfnorm}

MODELPRED = {}
for name,cfg in OPTIMIZEDEQS.items():
    entry     = SR_REGISTRY.get(name,{})
    form      = entry.get('form',cfg['form'])
    constants = entry.get('constants',cfg['init'])
    pnames    = used_predictors(form,VARS.keys())
    MODELPRED[name] = physical_prediction(eval_form(form,{p:VARS[p] for p in pnames},constants))

def load_predictions(name,split=SPLIT):
    path = os.path.join(PREDSDIR,f'{name}_{split}_predictions.nc')
    if not os.path.exists(path): return None
    with xr.open_dataset(path,engine='h5netcdf') as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims: da = da.mean('seed')
    pred = da.transpose('time','lat','lon').values.ravel()
    return pred if pred.shape[0]==obsflat.shape[0] else None

for name in {**PODRUNS,**NNRUNS}:
    pred = load_predictions(name)
    if pred is not None: MODELPRED[name] = pred

valid = np.isfinite(obsflat)
for arr in VARS.values(): valid &= np.isfinite(arr)

print('Loaded:', sorted(MODELPRED.keys()))
for name,pred in MODELPRED.items():
    print(f'  {LABELS.get(name,name):20s}  R²={r2(obsflat[valid],pred[valid]):.3f}')

## 1. Orographic maps
SDO marks the Western Ghats, Himalayan foothills, and the Eastern Ghats. ASH (if loaded) adds the large-scale elevation signal.

In [ ]:
panels = [('SDO (m)', SDO_MAP)]
if ASH_MAP is not None:
    panels.append(('ASH (m)', ASH_MAP))

fig,axs = pplt.subplots(nrows=1,ncols=len(panels),proj='cyl',refwidth=3,share=False)
axs = np.atleast_1d(axs).ravel()
for ax,(title,arr) in zip(axs,panels):
    m = ax.pcolormesh(lon,lat,arr,cmap='terrain',vmin=0,vmax=arr.max(),
                      transform=ccrs.PlateCarree())
    ax.format(title=title,latlim=(lat.min(),lat.max()),lonlim=(lon.min(),lon.max()),
              coast=True,borders=False,grid=True,gridminor=False)
    fig.colorbar(m,loc='b',label=title,ax=ax,width=0.15)
pplt.show()

## 2. Skill gap stratified by terrain roughness (SDO)
Bin R² into SDO quartiles. If the gap between SR-HI and NN-GAUSS grows in high-SDO bins, orographic complexity is the likely culprit.

In [ ]:
def plot_r2_vs_orog(orog_flat, orog_label='SDO (m)',
                    names=None, nbins=10, minsamples=500):
    if names is None:
        names = [n for n in ['sr_hi','nn_gauss','sr_med','nn_bl'] if n in MODELPRED]
    obs   = obsflat[valid]
    orog  = orog_flat[valid]
    lo,hi = np.percentile(orog,[2,98])
    edges = np.linspace(lo,hi,nbins+1)
    xc    = 0.5*(edges[:-1]+edges[1:])
    idxs  = np.clip(np.digitize(orog,edges)-1,0,nbins-1)

    fig,ax = pplt.subplots(refwidth=3,refheight=2)
    for name in names:
        pred = MODELPRED[name][valid]
        r2s  = []
        for i in range(nbins):
            idx = idxs==i
            r2s.append(r2(obs[idx],pred[idx]) if idx.sum()>=minsamples else np.nan)
        ax.plot(xc,r2s,color=COLORS[name],linewidth=1.5,marker='o',
                markersize=3,label=LABELS[name])
    ax.format(xlabel=orog_label,ylabel='R²',
              title=f'Skill vs. {orog_label}',grid=False)
    ax.legend(loc='ur',ncols=1)
    pplt.show()

plot_r2_vs_orog(sdoraw,'SDO — physical units (m)')
if ASH_FLAT is not None:
    plot_r2_vs_orog(ASH_FLAT,'ASH — mean elevation (m)')

## 3. Spatial residual maps with terrain contours
Time-mean (predicted − observed), with SDO contours overlaid, for SR-HI and NN-GAUSS. Co-location of bias with high SDO = orographic signal the model misses.

In [ ]:
def to_map(flat):
    return np.nanmean(flat.reshape(ntime,nlat,nlon),axis=0)

obs_map  = to_map(obsflat)
sdo_map  = SDO_MAP   # already (nlat,nlon)

names = [n for n in ['sr_hi','nn_gauss','sr_med'] if n in MODELPRED]
fig,axs = pplt.subplots(nrows=1,ncols=len(names),proj='cyl',refwidth=2.5,share=True)
axs = np.atleast_1d(axs).ravel()
m = None
for ax,name in zip(axs,names):
    resid = to_map(MODELPRED[name]) - obs_map
    m = ax.pcolormesh(lon,lat,resid,cmap='DryWet',vmin=-0.5,vmax=0.5,
                      extend='both',transform=ccrs.PlateCarree())
    # Terrain contours
    ax.contour(lon,lat,sdo_map,levels=[100,300,500],
               colors='k',linewidths=0.5,transform=ccrs.PlateCarree())
    ax.format(title=LABELS[name],latlim=(lat.min(),lat.max()),
              lonlim=(lon.min(),lon.max()),coast=True,borders=False,
              grid=True,gridminor=False)
fig.colorbar(m,loc='b',label='Predicted $-$ Observed (mm)',width=0.15,length=0.7)
pplt.show()

## 4. Wind + topography: where mechanical forcing matters

Orographic precipitation is inherently directional: the same ridge produces very different rainfall on its windward vs. leeward flanks depending on the low-level flow direction. Column-integral thermodynamic variables (RH, θₑ, θₑ*) carry no information about the **direction** of moisture transport — they only encode how much moisture and instability is present, not whether it is being forced upward by terrain.

To test this: we load 850 hPa winds (u, v) from ERA5, compute wind direction relative to local terrain slope, and check whether SR-HI's residuals are largest at grid points with strong upslope flow (windward) vs. downslope (leeward).

In [ ]:
# Load 850 hPa winds from ERA5 raw files
# ERA5 filenames are approximate — adjust to match your RAWDIR naming convention
U850_MAP = None
V850_MAP = None

for ucand,vcand in [
    ('ERA5_u_component_of_wind.nc','ERA5_v_component_of_wind.nc'),
    ('ERA5_u850.nc','ERA5_v850.nc'),
    ('ERA5_wind_u.nc','ERA5_wind_v.nc'),
]:
    up = os.path.join(RAWDIR,ucand)
    vp = os.path.join(RAWDIR,vcand)
    if os.path.exists(up) and os.path.exists(vp):
        with xr.open_dataset(up) as ds:
            uvar = [v for v in ds.data_vars][0]
            u = ds[uvar].sel(level=850,method='nearest').mean('time') if 'level' in ds.dims else ds[uvar].mean('time')
            U850_MAP = u.interp(latitude=lat[::-1],longitude=lon).values[::-1]
        with xr.open_dataset(vp) as ds:
            vvar = [v for v in ds.data_vars][0]
            v = ds[vvar].sel(level=850,method='nearest').mean('time') if 'level' in ds.dims else ds[vvar].mean('time')
            V850_MAP = v.interp(latitude=lat[::-1],longitude=lon).values[::-1]
        print(f'Loaded 850 hPa winds from {ucand}/{vcand}')
        break

if U850_MAP is None:
    print('850 hPa wind files not found in RAWDIR.')
    print('To enable wind analysis, add ERA5 u/v at 850 hPa to the data pipeline.')

In [ ]:
def plot_wind_and_terrain():
    """Climatological 850 hPa winds overlaid on SDO and SR-HI mean residual."""
    if U850_MAP is None:
        print('Wind data not available — skipping.')
        return
    srhi_resid = to_map(MODELPRED['sr_hi']) - obs_map if 'sr_hi' in MODELPRED else None

    skip = max(1,nlat//12)  # subsample arrows for readability
    lo_  = lat[::skip]
    lo2_ = lon[::skip]
    uu   = U850_MAP[::skip,::skip]
    vv   = V850_MAP[::skip,::skip]

    nplots = 1 + (srhi_resid is not None)
    fig,axs = pplt.subplots(nrows=1,ncols=nplots,proj='cyl',refwidth=3,share=False)
    axs = np.atleast_1d(axs).ravel()

    # Panel 1: SDO + winds
    m1 = axs[0].pcolormesh(lon,lat,sdo_map,cmap='terrain',vmin=0,vmax=sdo_map.max(),
                            transform=ccrs.PlateCarree())
    axs[0].quiver(lo2_,lo_,uu,vv,transform=ccrs.PlateCarree(),
                  color='white',scale=150,width=0.003)
    axs[0].format(title='SDO + 850 hPa Climatological Winds',
                  latlim=(lat.min(),lat.max()),lonlim=(lon.min(),lon.max()),
                  coast=True,borders=False,grid=True,gridminor=False)
    fig.colorbar(m1,loc='b',label='SDO (m)',ax=axs[0],width=0.15)

    # Panel 2: SR-HI residual + winds
    if srhi_resid is not None:
        m2 = axs[1].pcolormesh(lon,lat,srhi_resid,cmap='DryWet',vmin=-0.5,vmax=0.5,
                                extend='both',transform=ccrs.PlateCarree())
        axs[1].contour(lon,lat,sdo_map,levels=[100,300,500],
                       colors='k',linewidths=0.5,transform=ccrs.PlateCarree())
        axs[1].quiver(lo2_,lo_,uu,vv,transform=ccrs.PlateCarree(),
                      color='k',scale=150,width=0.003,alpha=0.6)
        axs[1].format(title='SR-HI Residual + Winds + Terrain',
                      latlim=(lat.min(),lat.max()),lonlim=(lon.min(),lon.max()),
                      coast=True,borders=False,grid=True,gridminor=False)
        fig.colorbar(m2,loc='b',label='Predicted $-$ Observed (mm)',ax=axs[1],width=0.15)
    pplt.show()

plot_wind_and_terrain()

## 5. Upslope flow index vs. model skill

Compute the dot product of the 850 hPa wind vector with the local terrain gradient (∇h). Positive values = upslope (windward) flow → enhanced orographic ascent. Negative = downslope (leeward). Bin R² by upslope index to test whether SR-HI fails preferentially where mechanical forcing is strongest.

In [ ]:
def compute_upslope_index(ash_map, u_map, v_map):
    """Dot product of (u,v) with terrain gradient (dh/dx, dh/dy)."""
    dhdx = np.gradient(ash_map, axis=1)   # ∂h/∂lon
    dhdy = np.gradient(ash_map, axis=0)   # ∂h/∂lat
    return u_map*dhdx + v_map*dhdy        # (nlat,nlon)

def plot_r2_vs_upslope(names=None, nbins=10, minsamples=500):
    if U850_MAP is None or ASH_MAP is None:
        print('Need both wind data and ASH for upslope index — skipping.')
        return
    if names is None:
        names = [n for n in ['sr_hi','nn_gauss','sr_med'] if n in MODELPRED]

    upsmap  = compute_upslope_index(ASH_MAP,U850_MAP,V850_MAP)
    upsflat = np.tile(upsmap,(ntime,1,1)).ravel()[valid]
    obs     = obsflat[valid]

    lo,hi  = np.percentile(upsflat,[5,95])
    edges  = np.linspace(lo,hi,nbins+1)
    xc     = 0.5*(edges[:-1]+edges[1:])
    idxs   = np.clip(np.digitize(upsflat,edges)-1,0,nbins-1)

    fig,ax = pplt.subplots(refwidth=3,refheight=2)
    ax.axvline(0,color='gray',linewidth=0.8,linestyle='--')
    for name in names:
        pred = MODELPRED[name][valid]
        r2s  = [r2(obs[idxs==i],pred[idxs==i]) if (idxs==i).sum()>=minsamples else np.nan
                for i in range(nbins)]
        ax.plot(xc,r2s,color=COLORS[name],linewidth=1.5,marker='o',markersize=3,label=LABELS[name])
    ax.format(xlabel='Upslope flow index (u·∇h)',ylabel='R²',
              title='Skill vs. Upslope Flow (mechanical forcing)',grid=False)
    ax.legend(loc='ur',ncols=1)
    pplt.show()

plot_r2_vs_upslope()

## Discussion: Why orographic precipitation is hard in this framework

The inputs available to SR and NN models are **column-integral thermodynamic quantities**: kernel-integrated RH, θₑ, θₑ*, and near-surface flux variables (LF, SHF, LHF). These encode *how much* moisture and instability is present in a column, but carry no information about:

1. **Wind direction relative to topography** — the same column of moist air produces vastly different precipitation depending on whether it is being forced upslope (windward ascent) or descending on the leeward side.

2. **Sub-grid terrain geometry** — SDO partially captures terrain roughness, but not ridge orientation, valley channelling, or aspect. The Western Ghats produce a sharp precipitation shadow that requires knowing the gradient direction of the escarpment relative to the prevailing south-westerly flow.

3. **Mechanical triggering vs. thermodynamic potential** — SR-HI's equation `cube(max(rh, θₑ − 1.47θₑ* − c))` is essentially a buoyancy threshold: it fires when the column is sufficiently unstable. Orographic precipitation can occur even in thermodynamically marginal columns if mechanical ascent is strong enough. This is a different physical regime that a thermodynamic-only equation cannot represent.

**What would actually help:** Adding low-level wind speed and direction (e.g., 850 hPa u, v) as inputs, computing an upslope flow index, or including moisture flux convergence (∇·(qV)) as a predictor. These would let the model distinguish windward from leeward ascent. The limitation is that they require adding dynamic variables to the data pipeline, which we have not yet done.